# CMS real-world readmission data: preliminary EDA

Explore CMS hospital/state files, California all-cause readmission strata, and (if available) aggregated synthetic stays. These sources have different units, populations, definitions, and reporting periods. Comparisons are descriptive only: no patient-level linkage or model validation is possible.

Raw data remains in git-ignored `archive/real_world/`; this notebook writes no derived data files.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
DATA_DIR = ROOT / 'archive' / 'real_world'
assert DATA_DIR.exists(), f'Missing {DATA_DIR}; run from repo root or notebooks/'
print(DATA_DIR)

## 1. Load files and preserve suppression
Read as strings first. Suppressed CMS values are missing, not zero. Preserve facility identifiers as strings so leading zeros survive.

In [ ]:
def find_file(pattern):
    found = sorted(DATA_DIR.glob(pattern))
    return found[0] if found else None
def read_file(path):
    if path is None: return None
    print(path.name)
    return pd.read_csv(path, dtype='string', low_memory=False)
cms_hospital_raw = read_file(find_file('Unplanned_Hospital_Visits-Hospital.csv'))
cms_state_raw = read_file(find_file('Unplanned_Hospital_Visits-State.csv'))
hrrp_raw = read_file(find_file('FY_*_Hospital_Readmissions_Reduction_Program_Hospital.csv'))
ca_candidates = [p for p in DATA_DIR.glob('all-cause-unplanned-30-day-hospital-readmission-rate-california-*.csv') if 'dictionary' not in p.name.casefold()]
ca_raw = read_file(sorted(ca_candidates)[0] if ca_candidates else None)
pch_hospital_raw = read_file(find_file('PCH_Complications_Unplanned_Hospital_Visits_HOSPITAL.csv'))
for name, d in [('CMS hospital',cms_hospital_raw),('CMS state',cms_state_raw),('HRRP',hrrp_raw),('California',ca_raw),('PCH hospital',pch_hospital_raw)]:
    if d is not None: print(name, d.shape); display(d.head(2))

## 2. CMS hospital readmission measures
Filter to condition-specific 30-day measures and hospital-wide readmission. Keep source reporting dates and footnotes. CMS scores are generally in percent units; do not mix them with HRRP excess readmission ratios.

In [ ]:
def numeric(s):
    return pd.to_numeric(s.replace({'':pd.NA,'Not Available':pd.NA,'Not Applicable':pd.NA,'Too Few to Report':pd.NA,'N/A':pd.NA}), errors='coerce')
hospital_readmit = None
if cms_hospital_raw is not None:
    required = {'Facility ID','Facility Name','State','Measure ID','Measure Name','Score'}
    missing = required - set(cms_hospital_raw.columns)
    if missing: raise ValueError(f'Missing CMS columns: {missing}')
    hospital_readmit = cms_hospital_raw[cms_hospital_raw['Measure ID'].str.startswith(('READM_30','READM-30','Hybrid_HWR'),na=False)].copy()
    hospital_readmit = hospital_readmit.rename(columns={'Facility ID':'facility_id','Facility Name':'facility_name','State':'state','Measure ID':'measure_id','Measure Name':'measure_name','Score':'score','Start Date':'start_date','End Date':'end_date'})
    hospital_readmit['facility_id'] = hospital_readmit['facility_id'].str.strip().str.zfill(6)
    hospital_readmit['score'] = numeric(hospital_readmit['score'])
    display(hospital_readmit.groupby(['measure_id','measure_name'],dropna=False).agg(rows=('facility_id','size'),facilities=('facility_id','nunique'),scores_available=('score','count'),median_score=('score','median'),mean_score=('score','mean')).round(2))
    co = hospital_readmit[hospital_readmit.state.str.upper().eq('CO')]
    print('Colorado rows:',len(co),'facilities:',co.facility_id.nunique())
    display(co[['facility_id','facility_name','measure_id','score','Compared to National','start_date','end_date','Footnote']].sort_values(['measure_id','score'],na_position='last'))

## 3. Exploratory clustering of hospital profiles
Cluster facilities by their reported readmission scores only. This is descriptive pattern-finding, not a clinical risk model. We require measure coverage, median-impute only for this exploratory step, standardize, and report silhouette scores. Cluster labels are not inherently meaningful; CMS measures have different populations and definitions.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
if hospital_readmit is not None:
    hospital_readmit['period'] = hospital_readmit.get('start_date',pd.Series('',index=hospital_readmit.index)).fillna('')+' to '+hospital_readmit.get('end_date',pd.Series('',index=hospital_readmit.index)).fillna('')
    prof = hospital_readmit.pivot_table(index=['facility_id','facility_name','state','period'],columns='measure_id',values='score',aggfunc='first')
    prof = prof.dropna(axis=1,thresh=max(10,int(.25*len(prof))))
    prof = prof.dropna(axis=0,thresh=max(2,int(.60*prof.shape[1])))
    print('Profiles:',prof.shape,'Measures:',list(prof.columns))
    if len(prof)>=10 and prof.shape[1]>=2:
        X=prof.apply(lambda c:c.fillna(c.median()))
        Z=StandardScaler().fit_transform(X)
        rows=[]
        for k in range(2,min(7,len(X)-1)):
            labels=KMeans(n_clusters=k,random_state=0,n_init=20).fit_predict(Z)
            rows.append({'k':k,'silhouette':silhouette_score(Z,labels),'smallest_cluster':pd.Series(labels).value_counts().min()})
        scores=pd.DataFrame(rows); display(scores.round(3))
        k=int(scores.sort_values('silhouette',ascending=False).iloc[0].k)
        prof['cluster']=KMeans(n_clusters=k,random_state=0,n_init=20).fit_predict(Z)
        display(prof.groupby('cluster')[list(X.columns)].median().round(2))
        display(prof.reset_index().groupby(['state','cluster']).size().unstack(fill_value=0).head(15))
    else: print('Insufficient measure/facility coverage for a multi-measure cluster analysis.')

## 4. State summary and HRRP
CMS State reports counts of hospitals rated better/same/worse/too few; it is not a state readmission rate. HRRP reports condition-specific excess readmission ratios (ERR), predicted and expected rates. ERR is neither a raw probability nor the payment penalty. Keep the separate reporting periods visible.

In [ ]:
if cms_state_raw is not None:
    sr=cms_state_raw[cms_state_raw['Measure ID'].str.startswith(('READM_30','READM-30','Hybrid_HWR'),na=False)].copy()
    count_cols=[c for c in sr.columns if c.startswith('Number of Hospitals')]
    for c in count_cols: sr[c]=numeric(sr[c])
    display(sr[sr.State.eq('CO')])
    display(sr.groupby(['Measure ID','Measure Name'])[count_cols].sum(min_count=1).head(20))
if hrrp_raw is not None:
    h=hrrp_raw.rename(columns={'Facility ID':'facility_id','Facility Name':'facility_name','State':'state','Excess Readmission Ratio':'err','Predicted Readmission Rate':'predicted_rate','Expected Readmission Rate':'expected_rate','Number of Discharges':'discharges','Number of Readmissions':'readmissions'}).copy()
    h['facility_id']=h.facility_id.str.strip().str.zfill(6)
    for c in ['err','predicted_rate','expected_rate','discharges','readmissions']: h[c]=numeric(h[c])
    display(h[h.state.eq('CO')].sort_values('Measure Name')[['facility_id','facility_name','Measure Name','err','predicted_rate','expected_rate','discharges','readmissions','Start Date','End Date']])

## 5. California all-cause trends and strata
California publishes counts/rates by year, demographic strata, payer, and county. Use this for broad pattern comparison only: it is California, includes multiple payer groups, and its all-cause unadjusted measure differs from the synthetic Medicare cohort and CMS hospital measures.

In [ ]:
ca=None
if ca_raw is not None:
    ca=ca_raw.rename(columns={c:c.strip() for c in ca_raw.columns}).copy()
    ca['Year']=pd.to_numeric(ca['Year'],errors='coerce')
    rate_cols=[c for c in ca.columns if 'Readmission Rate' in c]
    for c in rate_cols: ca[c+'_pct']=pd.to_numeric(ca[c].str.replace('%','',regex=False),errors='coerce')
    state_ca=ca[ca.County.eq('State')].copy()
    display(state_ca.groupby(['Year','Strata','Strata Name'],dropna=False)[[c+'_pct' for c in rate_cols]].first().tail(40))
    overall=state_ca[state_ca['Strata Name'].str.casefold().isin(['overall','all'])]
    if not overall.empty:
        ax=overall.sort_values('Year').plot(x='Year',y=[c+'_pct' for c in rate_cols],marker='o',figsize=(9,4),title='California all-cause unplanned 30-day readmissions')
        ax.set_ylabel('Rate (%)'); ax.grid(alpha=.25); plt.show()
    else: print('Inspect strata names to identify statewide total; no Overall/All label found.')

## 6. Aggregate synthetic stays for a cautious comparison
This runs if cleaned Parquet tables exist. It compares broad age, sex, and race strata with California statewide strata where possible. Because California’s population, period, and measure definitions differ, these are directional pattern comparisons, not validation. The CMS synthetic state code is not used as real geography. For a Colorado Medicare geography comparison, use CMS Geographic Variation PUF separately; the national CMS hospital file already contains Colorado facilities.

In [ ]:
try:
    from hospital_readmissions.data import load_readmit
    syn=load_readmit(renamed=True)
    syn['age_group']=pd.cut(syn.age_at_admission,bins=[17,44,64,120],labels=['18 to 44 years','45 to 64 years','65 years and above'],include_lowest=True)
    syn['sex_group']=syn.sex.astype('string').str.upper().map({'M':'Male','F':'Female','1':'Male','2':'Female'})
    syn['race_group']=syn.race.astype('string').str.strip()
    syn_summary=syn.groupby(['age_group','sex_group','race_group'],observed=True).readmitted_30_days.agg(stays='size',readmissions='sum',rate='mean').reset_index()
    syn_summary['rate_pct']=100*syn_summary.rate
    print(f'{len(syn):,} eligible stays; overall rate {syn.readmitted_30_days.mean():.2%}')
    display(syn_summary.sort_values('stays',ascending=False).head(30).round(2))
except (FileNotFoundError,ImportError) as exc:
    print('Synthetic cleaned Parquet tables/package unavailable. Run data-cleaning notebook and pip install -e . to enable this section.')
    print(exc)

In [ ]:
# Side-by-side age-stratum comparison: synthetic 2008-2010 pooled stays vs
# latest year available in the California file. Rates are unadjusted and definitions differ.
if ca is not None and 'syn_summary' in globals():
    ca_age=ca[(ca.County.eq('State')) & (ca.Strata.eq('Age'))].copy()
    ca_age=ca_age[ca_age.Year.eq(ca_age.Year.max())][['Strata Name']+[c+'_pct' for c in rate_cols if c+'_pct' in ca_age]].copy()
    ca_age=ca_age.rename(columns={'Strata Name':'age_group'})
    syn_age=syn.groupby('age_group',observed=True).readmitted_30_days.agg(stays='size',readmissions='sum',synthetic_rate='mean').reset_index()
    syn_age['synthetic_rate_pct']=100*syn_age.synthetic_rate
    age_compare=ca_age.merge(syn_age,on='age_group',how='inner')
    display(age_compare.round(2))
    if not age_compare.empty:
        ax=age_compare.set_index('age_group')[['synthetic_rate_pct']+[c+'_pct' for c in rate_cols if c+'_pct' in age_compare]].plot(kind='bar',figsize=(9,4),title='Age-stratified 30-day readmission: synthetic vs California')
        ax.set_ylabel('Rate (%)'); ax.set_xlabel('Age group'); ax.legend(['Synthetic pooled 2008-2010']+[f'California {c}' for c in rate_cols if c+'_pct' in age_compare]); ax.grid(axis='y',alpha=.25); plt.xticks(rotation=0); plt.show()
    else: print('No exact age-label overlap. Inspect California categories and map labels explicitly before comparing.')
else: print('Run synthetic and California sections first to create age summaries.')

## Preliminary interpretation checklist

1. Inspect the California methodology/data dictionary to confirm rate columns and year coverage before presenting trends.
2. Compare synthetic and California age/sex/race patterns only for overlapping groups, always showing denominators.
3. Use CMS hospital records for Colorado facility descriptions; use the CMS Medicare Geographic Variation PUF for state/county context if needed.
4. Check whether clusters persist across measure subsets/reporting periods before naming profiles.
5. Do not interpret CMS State hospital counts as rates, HRRP ERR as readmission probability, or any aggregate result causally.